In [1]:
import os
import h5py
import pywt
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from typing import List, Tuple
from scipy.stats import entropy
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from utils import process_file, label_lab, experiment_based_classification

In [2]:
## Load stimuli
file = "../data/1D_brownian_stimuli_responses/MR-0644_brownian_stimuli.h5"
ts = []
with h5py.File(file, "r") as f:
    for key in f.keys():
        ts.append(f[key]["data"][:])

## Load responses
path = '../data/1D_brownian_stimuli_responses/'

# List all files in the directory
files = os.listdir(path)

# Filter only files (exclude directories)
file_names = [f for f in files if os.path.isfile(os.path.join(path, f))]
file_names = [f for f in file_names if "brownian.h5" in f]
print("Files in the directory:", file_names)

def process_file(file_path):
    with h5py.File(path+file_path, 'r') as File:
        events_means = [[] for _ in range(6)]
        for electrode in list(File.keys()):
            events = list(File[electrode].keys())
            for event in events:
                data = File[electrode][f'{event}']['data'][:]
                events_means[int(event[6:])-1].append(data)
        final_means = [np.mean(event, axis=0) for event in events_means]
    return final_means

df = pd.DataFrame({"H":[i for i in range(6)]})
for file in file_names:
    means = process_file(file)
    df[file[:-12]] = means
df = df.drop(columns=["H"])
df = df.T

Files in the directory: ['MR-0644_brownian.h5', 'MR-0645_brownian.h5', 'MR-0648_brownian.h5', 'MR-0649_brownian.h5', 'MR-0654-t1_brownian.h5', 'MR-0654-t2_brownian.h5', 'MR-0655_brownian.h5', 'MR-0656_brownian.h5', 'MR-0657-t2_brownian.h5', 'MR-0658_brownian.h5', 'MR-0659-t1_brownian.h5', 'MR-0659-t2_brownian.h5', 'MR-0661_brownian.h5', 'MR-0662_brownian.h5', 'MR-0663_brownian.h5', 'MR-0665_brownian.h5', 'MR-0667_brownian.h5', 'MR-0668_brownian.h5', 'MR-0669_brownian.h5', 'MR-0674_brownian.h5', 'MR-0676_brownian.h5', 'MR-0677_brownian.h5', 'MR-0678_brownian.h5', 'MR-0679_brownian.h5', 'MR-0680_brownian.h5', 'MR-0681_brownian.h5', 'MR-0682_brownian.h5']


In [5]:
groups = {
    "5xFAD_3mF": ['MR-0644', 'MR-0645', 'MR-0648', 'MR-0649'],
    "5xFAD_3mM": ['MR-0661','MR-0663','MR-0667'],
    "5xFAD_6mF": ['MR-0659-t1', 'MR-0659-t2', 'MR-0676'],
    "5xFAD_6mM": ['MR-0657-t2', 'MR-0674', 'MR-0656'],
    "WT_3mF": ['MR-0677', 'MR-0678'],
    "WT_3mM": ['MR-0662', 'MR-0665', 'MR-0668', 'MR-0669'],
    "WT_6mF": ['MR-0679', 'MR-0680'],
    "WT_6mM": ['MR-0655', 'MR-0658', 'MR-0654-t1', 'MR-0654-t2', 'MR-0687-t1']
}

colors = {}
for key in groups.keys():
    for value in groups[key]:
        if "5xFAD_3m" in key:
            colors[value] = "r"
        elif "5xFAD_6m" in key:
            colors[value] = "b"
        elif "WT_3m" in key:
            colors[value] = "g"
        elif "WT_6m" in key:
            colors[value] = "m"
        else:
            colors[value] = "k"

for value in df.index:
    if value != "H":
        if value not in colors.keys():
            colors[value] = "k"

def age(ind):
    if '3m' in ind:
        return 'young'
    else:
        return 'old'
    
def condition(ind):
    if '5xFAD' in ind:
        return '5xFAD'
    else:
        return 'Wild-Type'

wavelet_props = df.copy()
kl_stats = df.copy()

def label_lab(a):
    for key in groups.keys():
        if a in groups[key]:
            return key
        

# Group by color (excluding black)
color_groups = {}
for i in range(0,len(df.index)):
    color = colors[df.index[i]]
    if color != "black":
        color_groups.setdefault(color, []).append(i)

selected_colors = ['g','m', 'r', 'b','k']#list(color_groups.keys())[:4]
labels_sujetos = []
for ex in range(6):
    #fig, ax = plt.subplots(1, figsize=(8, 5))
    
    stimuli_profile = pywt.wavedec(ts[ex], wavelet='db2')
    stimuli_approx = stimuli_profile[0]
    stimuli_total_energy = np.sum(np.square(stimuli_approx))
    stimuli_details = [np.sum(np.square(stimuli_approx))]  # start with approximation energy
    for level_dict in stimuli_profile[1:]:
        level_energy = np.sum([np.sum(np.square(arr)) for arr in level_dict])
        stimuli_total_energy += level_energy
        stimuli_details.append(level_energy)

    stimuli_proportions = [e / stimuli_total_energy for e in stimuli_details]



    i=0
    for color in selected_colors:
        indices = color_groups[color]
        all_props = []

        for indice in indices:
            corr = np.abs(df.at[df.index[indice], ex])
            coeffs = pywt.wavedec(corr, wavelet='db2')
            approx = coeffs[0]

            total_energy = np.sum(np.square(approx))
            details = [np.sum(np.square(approx))]  # start with approximation energy
            for level_dict in coeffs[1:]:
                level_energy = np.sum([np.sum(np.square(arr)) for arr in level_dict])
                total_energy += level_energy
                details.append(level_energy)

            proportions = [e / total_energy for e in details]
            wavelet_props.at[df.index[indice], ex] = proportions
            kl_stats.at[df.index[indice], ex] = entropy(stimuli_proportions,proportions)
            if ex == 1:
                l = label_lab(df.index[indice])
                if l!= None:
                    labels_sujetos.append(l[:-1])
                else:
                    labels_sujetos.append('no class')

entropy_stats = wavelet_props.copy()
for column in entropy_stats.columns:
    if column != "type":
        entropy_stats[column] =  wavelet_props[column].apply(entropy)

wavelet_props['type'] = labels_sujetos
entropy_stats['type'] = labels_sujetos
kl_stats['type'] = labels_sujetos

wavelet_props = wavelet_props[wavelet_props['type'] != "no class"]
entropy_stats = entropy_stats[entropy_stats['type'] != "no class"]
kl_stats = kl_stats[kl_stats['type'] != "no class"]

for column in entropy_stats.columns:
    if column != "type":
        wavelet_props[column] =  wavelet_props[column].apply(lambda x: x[3])

In [4]:
experiment_based_classification(wavelet_props,age)
print("== entropy ==")
experiment_based_classification(entropy_stats,age)
print("== kl div ==")
experiment_based_classification(kl_stats,age)

LOOCV Accuracy 0: 8.00%
LOOCV Accuracy 1: 52.00%
LOOCV Accuracy 2: 48.00%
LOOCV Accuracy 3: 60.00%
LOOCV Accuracy 4: 48.00%
LOOCV Accuracy 5: 36.00%
== entropy ==
LOOCV Accuracy 0: 36.00%
LOOCV Accuracy 1: 68.00%
LOOCV Accuracy 2: 76.00%
LOOCV Accuracy 3: 56.00%
LOOCV Accuracy 4: 52.00%
LOOCV Accuracy 5: 40.00%
== kl div ==
LOOCV Accuracy 0: 16.00%
LOOCV Accuracy 1: 40.00%
LOOCV Accuracy 2: 64.00%
LOOCV Accuracy 3: 76.00%
LOOCV Accuracy 4: 52.00%
LOOCV Accuracy 5: 68.00%


{'0': 0.16, '1': 0.4, '2': 0.64, '3': 0.76, '4': 0.52, '5': 0.68}

In [6]:
experiment_based_classification(wavelet_props,condition)
print("== entropy ==")
experiment_based_classification(entropy_stats,condition)
print("== kl div ==")
experiment_based_classification(kl_stats,condition)

LOOCV Accuracy 0: 68.00%
LOOCV Accuracy 1: 52.00%
LOOCV Accuracy 2: 32.00%
LOOCV Accuracy 3: 64.00%
LOOCV Accuracy 4: 40.00%
LOOCV Accuracy 5: 56.00%
== entropy ==
LOOCV Accuracy 0: 52.00%
LOOCV Accuracy 1: 20.00%
LOOCV Accuracy 2: 60.00%
LOOCV Accuracy 3: 32.00%
LOOCV Accuracy 4: 48.00%
LOOCV Accuracy 5: 24.00%
== kl div ==
LOOCV Accuracy 0: 60.00%
LOOCV Accuracy 1: 32.00%
LOOCV Accuracy 2: 48.00%
LOOCV Accuracy 3: 60.00%
LOOCV Accuracy 4: 32.00%
LOOCV Accuracy 5: 64.00%


{'0': 0.6, '1': 0.32, '2': 0.48, '3': 0.6, '4': 0.32, '5': 0.64}

## coactiv

In [7]:
coactiv_stats = df.copy()
        
# Group by color (excluding black)
color_groups = {}
for i in range(0,len(df.index)):
    color = colors[df.index[i]]
    if color != "black":
        color_groups.setdefault(color, []).append(i)

selected_colors = ['g','m', 'r', 'b','k']#list(color_groups.keys())[:4]
labels_sujetos = []
for ex in range(6):
    #fig, ax = plt.subplots(1, figsize=(8, 5))
    
    i=0
    for color in selected_colors:
        indices = color_groups[color]
        all_props = []

        for indice in indices:
            corr = np.multiply(ts[ex],np.abs(df.at[df.index[indice], ex])[:7500])
            coeffs = pywt.wavedec(corr, wavelet='db2')
            approx = coeffs[0]

            total_energy = np.sum(np.square(approx))
            details = [np.sum(np.square(approx))]  # start with approximation energy
            for level_dict in coeffs[1:]:
                level_energy = np.sum([np.sum(np.square(arr)) for arr in level_dict])
                total_energy += level_energy
                details.append(level_energy)

            proportions = [e / total_energy for e in details]
            coactiv_stats.at[df.index[indice], ex] = proportions
            if ex == 1:
                l = label_lab(df.index[indice])
                if l!= None:
                    labels_sujetos.append(l[:-1])
                else:
                    labels_sujetos.append('no class')


coactiv_stats['type'] = labels_sujetos
coactiv_stats = coactiv_stats[coactiv_stats["type"]!="no class"]


for column in entropy_stats.columns:
    if column in [0,1,3,4]:
        coactiv_stats[column] =  coactiv_stats[column].apply(lambda x: x[3])
    elif column in [2,5]:
        coactiv_stats[column] =  coactiv_stats[column].apply(lambda x: x[6])

In [8]:
experiment_based_classification(coactiv_stats,age)
print("== cond ==")
experiment_based_classification(coactiv_stats,condition)


LOOCV Accuracy 0: 28.00%
LOOCV Accuracy 1: 48.00%
LOOCV Accuracy 2: 72.00%
LOOCV Accuracy 3: 36.00%
LOOCV Accuracy 4: 44.00%
LOOCV Accuracy 5: 64.00%
== cond ==
LOOCV Accuracy 0: 68.00%
LOOCV Accuracy 1: 52.00%
LOOCV Accuracy 2: 52.00%
LOOCV Accuracy 3: 68.00%
LOOCV Accuracy 4: 48.00%
LOOCV Accuracy 5: 60.00%


{'0': 0.68, '1': 0.52, '2': 0.52, '3': 0.68, '4': 0.48, '5': 0.6}